In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def add_label(label, data):
    return {x: f"{x}{label}" for x in data}

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "green_algae_mit",
    "green_algae_plt",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
]

polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}

polarity_3bin = {
    "++": "same",
    "--": "same",
    "+-": "conv",
    "-+": "div",
}

group_titles = {
    "fungi_mit": "Fungi (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "plants_plt": "Plants (plastids)",
    "protists_plt": "Protists (plastids)",
    "green_algae_plt": "Green algae (plastids)",
}

label_2bin = set(polarity_2bin.values())
label_3bin = set(polarity_3bin.values())

COLOR_pct = "#009E73"  # teal green
COLOR_SAME = "#D55E00"  # vermillion
COLOR_CONV = "#56B4E9"  # sky blue
COLOR_DIV = "#03045E"  # deep navy
COLOR_BORDER = "#F0E442"  # yellow

In [ ]:
data_2bin = {}

for g in groups:
    print(f"Group: {g}")
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    dict_genome_size = dict(zip(tsv["AN"], tsv["Genome_length"]))
    igs = igs.join(
        tsv.select(["AN", "Genome_length"]),
        on="AN",
        how="left",
        validate="m:1",  # many-to-one: ensures tsv has unique AN values
    )

    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin")
    )

    med = igs.group_by(["AN", "polarity_2bin"]).agg(
        pl.col("Length").median().alias("median_bp"),
        pl.col("Length").count().alias("count"),
        pl.col("Length").sum().alias("total_bp"),
    )

    wide = med.pivot(values="median_bp", index="AN", on="polarity_2bin")
    wide = wide.rename(add_label("_median", label_2bin))

    wide_counts = med.pivot(values="count", index="AN", on="polarity_2bin")
    wide_counts = wide_counts.rename(add_label("_count", label_2bin))

    wide_totals = med.pivot(values="total_bp", index="AN", on="polarity_2bin")
    wide_totals = wide_totals.rename(add_label("_total", label_2bin))

    igs_sizes = igs.group_by("AN").agg(pl.col("Length").sum().alias("total_igs_size"))
    wide = (
        wide.join(wide_counts, on="AN", how="left", validate="1:1")
        .join(wide_totals, on="AN", how="left", validate="1:1")
        .join(igs_sizes, on="AN", how="left", validate="m:1")
        .join(tsv.select(["AN", "Genome_length"]), on="AN", how="left", validate="1:1")
    )
    wide = wide.with_columns(
        (pl.col("total_igs_size") / pl.col("Genome_length")).alias("noncoding_pct")
    )

    wide = wide.with_columns(
        (pl.col("opp_count") / (pl.col("opp_count") + pl.col("same_count"))).alias(
            "count_opp_pct"
        ),
        (pl.col("same_count") / (pl.col("opp_count") + pl.col("same_count"))).alias(
            "count_same_pct"
        ),
        (pl.col("opp_total") / (pl.col("same_total") + pl.col("opp_total"))).alias(
            "length_opp_pct"
        ),
        (pl.col("same_total") / (pl.col("same_total") + pl.col("opp_total"))).alias(
            "length_same_pct"
        ),
    )
    wide = wide.with_columns(
        (pl.col("noncoding_pct") * 100).alias("noncoding_pct"),
        (pl.col("count_opp_pct") * 100).alias("count_opp_pct"),
        (pl.col("count_same_pct") * 100).alias("count_same_pct"),
        (pl.col("length_opp_pct") * 100).alias("length_opp_pct"),
        (pl.col("length_same_pct") * 100).alias("length_same_pct"),
    )

    data_2bin[g] = wide

In [ ]:
# using data_dict, create a 4x2 grid of scatter plots
fig, axs = plt.subplots(4, 2, figsize=(15, 20))
axs = axs.flatten()

for i, g in enumerate(groups):
    wide = data_2bin[g]
    axs[i].scatter(
        wide["count_opp_pct"],
        wide["noncoding_pct"],
        color=COLOR_pct,
    )
    axs[i].set_xlabel("Proportion of Opposite Polarity Gene Pairs (%)", fontsize=13)
    axs[i].set_ylabel("Proportion of Noncoding DNA (%)", fontsize=13)
    axs[i].set_title(
        group_titles[g],
        fontsize=14,
    )
    axs[i].tick_params(axis="both", labelsize=13)

plt.suptitle(
    "Per-genome percentage of opposite polarity gene pairs vs. percentage of noncoding DNA",
    fontsize=16,
    fontweight="bold",
    y=1.001,
)
plt.tight_layout()
plt.savefig("supplemental_figure1.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# using data_dict, create a 4x2 grid of scatter plots
fig, axs = plt.subplots(4, 2, figsize=(15, 20))
axs = axs.flatten()

for i, g in enumerate(groups):
    wide = data_2bin[g]
    axs[i].scatter(wide["count_same_pct"], wide["noncoding_pct"], color=COLOR_SAME)
    axs[i].set_xlabel("Proportion of Same Polarity Gene Pairs (%)", fontsize=13)
    axs[i].set_ylabel("Proportion of Noncoding DNA (%)", fontsize=13)
    axs[i].set_title(
        group_titles[g],
        fontsize=14,
    )
    axs[i].tick_params(axis="both", labelsize=13)

plt.suptitle(
    "Per-genome percentage of same polarity gene pairs vs. percentage of noncoding DNA",
    fontsize=16,
    fontweight="bold",
    y=1.001,
)

plt.tight_layout()
plt.savefig("supplemental_figure2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(15, 20))
axs = axs.flatten()

for i, g in enumerate(groups):
    wide = data_2bin[g]

    above = wide["length_opp_pct"] > wide["count_opp_pct"]

    axs[i].scatter(
        wide.filter(~above)["count_opp_pct"],
        wide.filter(~above)["length_opp_pct"],
        color=COLOR_pct,
        label="Below diagonal",
    )
    axs[i].scatter(
        wide.filter(above)["count_opp_pct"],
        wide.filter(above)["length_opp_pct"],
        color=COLOR_pct,
        edgecolors=COLOR_BORDER,
        linewidths=0.8,
        zorder=3,
        label="Above diagonal",
    )

    axs[i].set_xlabel(
        "Proportion of Opposite Polarity Gene Pairs (count) (%)", fontsize=13
    )
    axs[i].set_ylabel("Proportion of Opposite Polarity IGR (length) (%)", fontsize=13)
    axs[i].set_title(group_titles[g], fontsize=14)
    axs[i].tick_params(axis="both", labelsize=13)
    axs[i].plot([0, 100], [0, 100], color="black", linestyle="--")

plt.suptitle(
    "Per-genome percentage of opposite polarity gene pairs (count) vs. per-genome percentage of opposite polarity IGR (length)",
    fontsize=16,
    fontweight="bold",
    y=1.001,
)

plt.tight_layout()
plt.savefig("supplemental_figure3.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
group_rows = []

for g in groups:
    print(group_titles[g])
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    igs = igs.join(
        tsv.select(["AN", "Genome_length"]),
        on="AN",
        how="left",
        validate="m:1",
    )

    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_3bin).alias("polarity_3bin")
    )

    med = igs.group_by(["AN", "polarity_3bin"]).agg(
        pl.col("Length").count().alias("count"),
        pl.col("Length").sum().alias("total_bp"),
    )

    wide_counts = med.pivot(values="count", index="AN", on="polarity_3bin")
    wide_counts = wide_counts.rename(add_label("_count", label_3bin))

    wide_totals = med.pivot(values="total_bp", index="AN", on="polarity_3bin")
    wide_totals = wide_totals.rename(add_label("_total", label_3bin))

    igs_sizes = igs.group_by("AN").agg(pl.col("Length").sum().alias("total_igs_size"))

    wide = wide_counts.join(wide_totals, on="AN", how="left", validate="1:1").join(
        igs_sizes, on="AN", how="left", validate="m:1"
    )

    # --- enrichment ratios ---
    sum_total = wide["total_igs_size"].sum()
    sum_counts = (
        wide["conv_count"].sum() + wide["div_count"].sum() + wide["same_count"].sum()
    )

    P_obs_conv = wide["conv_total"].sum() / sum_total
    P_obs_div = wide["div_total"].sum() / sum_total
    P_obs_same = wide["same_total"].sum() / sum_total

    P_exp_conv = wide["conv_count"].sum() / sum_counts
    P_exp_div = wide["div_count"].sum() / sum_counts
    P_exp_same = wide["same_count"].sum() / sum_counts

    E_conv = P_obs_conv / P_exp_conv
    E_div = P_obs_div / P_exp_div
    E_same = P_obs_same / P_exp_same

    group_rows.append({"group": g, "E_conv": E_conv, "E_div": E_div, "E_same": E_same})

    print(
        f"Observed pct (by length): Convergent: {P_obs_conv:.4f}, Divergent: {P_obs_div:.4f}, Same: {P_obs_same:.4f}"
    )
    print(
        f"Expected pct (by count):  Convergent: {P_exp_conv:.4f}, Divergent: {P_exp_div:.4f}, Same: {P_exp_same:.4f}"
    )
    print(
        f"Enrichment Ratios:        Convergent: {E_conv:.4f}, Divergent: {E_div:.4f}, Same: {E_same:.4f}"
    )
    print()

data_3bin = pl.DataFrame(group_rows)

In [ ]:
x = list(range(len(data_3bin)))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))

bars_conv = ax.bar(
    [i - width for i in x],
    data_3bin["E_conv"],
    width,
    label="Convergent",
    color=COLOR_CONV,
    zorder=3,
)
bars_div = ax.bar(
    x, data_3bin["E_div"], width, label="Divergent", color=COLOR_DIV, zorder=3
)
bars_same = ax.bar(
    [i + width for i in x],
    data_3bin["E_same"],
    width,
    label="Same",
    color=COLOR_SAME,
    zorder=3,
)

ax.axhline(
    1, color="black", linewidth=1.2, linestyle="--", zorder=2, label="E = 1 (null)"
)

ax.set_xticks(x)
ax.set_xticklabels(
    [group_titles[g].replace(" (", "\n(") for g in data_3bin["group"]],
    fontsize=10,
)
ax.set_ylabel("Enrichment ratio (E)", fontsize=11)
ax.set_ylim(
    0, max([*data_3bin["E_conv"], *data_3bin["E_div"], *data_3bin["E_same"]]) * 1.15
)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

legend_elements = [
    mpatches.Patch(color=COLOR_CONV, label="Convergent"),
    mpatches.Patch(color=COLOR_DIV, label="Divergent"),
    mpatches.Patch(color=COLOR_SAME, label="Same"),
    plt.Line2D(
        [0], [0], color="black", linestyle="--", linewidth=1.2, label="E = 1 (null)"
    ),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.set_title(
    "Enrichment ratios of convergent, divergent, and same polarity IGRs across groups",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

plt.tight_layout()
plt.savefig("supplemental_figure4.png", dpi=300, bbox_inches="tight")
plt.show()